# Projeto 3 — Análise exploratória e preparação (NWPU-RESISC45, 10 classes)

**Autores:** Paulo Henrico Rabelo (2312652) e Joey Alan (2320416)

Este notebook documenta o inventário, a integridade, as duplicatas, a divisão treino/validação/teste e a exploração visual.
A divisão é gerada por `scripts/preparar_dados.py` e **apenas lida** aqui. A exploração visual usa **somente o treino**;
validação e teste aparecem só em contagens, integridade e verificação de sobreposição. Os originais não são alterados.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as Show, display
from PIL import Image

from projeto3.classes import CLASSES, NOMES_PT

FIG = ROOT / "resultados" / "figuras"
FIG.mkdir(parents=True, exist_ok=True)
FONTE = "Fonte: elaborado pelos autores a partir do NWPU-RESISC45 (Cheng; Han; Lu, 2017)."
pd.set_option("display.width", 140)


def salvar(fig, nome):
    fig.text(0.01, 0.003, FONTE, fontsize=7, color="0.35")
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    fig.savefig(FIG / nome, dpi=150)
    plt.show()

particoes = pd.read_csv(ROOT / "dados/splits/particoes.csv")
inventario = json.loads((ROOT / "dados/splits/inventario.json").read_text(encoding="utf-8"))
treino = particoes[particoes["particao"] == "treino"].reset_index(drop=True)
print(f"{len(particoes)} imagens; origem: {inventario['origem']}; semente: {inventario['semente']}")

## 1. Inventário: arquivos, dimensões e canais

O script abre cada arquivo (`PIL.Image.verify` e uma segunda leitura), registra dimensões e modo de cor e calcula SHA-256.

In [ ]:
print("Dimensões encontradas:", inventario["dimensoes"])
print("Modos de cor:", inventario["modos_cor"])
print("Problemas de leitura/formato:", inventario["problemas"] or "nenhum")
print("Exclusões:", inventario["exclusoes"] or "nenhuma")
contagens = pd.crosstab(particoes["classe"], particoes["particao"]).reindex(index=CLASSES, columns=["treino", "validacao", "teste"])
contagens["total"] = contagens.sum(axis=1)
contagens.loc["TOTAL"] = contagens.sum()
contagens

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
base = np.zeros(len(CLASSES))
for particao, cor in zip(["treino", "validacao", "teste"], ["#4C72B0", "#DD8452", "#55A868"]):
    valores = contagens.loc[list(CLASSES), particao].to_numpy()
    ax.bar([NOMES_PT[c] for c in CLASSES], valores, bottom=base, label=particao, color=cor)
    base += valores
ax.set_ylabel("imagens")
ax.set_title("Imagens por classe e partição (70/15/15, estratificado, semente 42)")
ax.tick_params(axis="x", rotation=35)
ax.legend()
salvar(fig, "eda_contagens.png")

## 2. Duplicatas e versões da mesma imagem

* **Exatas:** mesmo SHA-256.
* **Quase-duplicatas:** dHash de 256 bits (imagem em cinza 17×17, diferença entre vizinhos horizontais) comparado nas
  8 orientações (rotações de 90° e espelhamentos); pares com distância de Hamming ≤ limiar são ligados por *union-find*.

Cada grupo é sorteado como uma unidade, então todas as versões caem na mesma partição. O histograma abaixo mostra a
distância de cada imagem ao vizinho mais próximo e justifica o limiar.

In [ ]:
q = inventario["quase_duplicatas"]
print(f"Duplicatas exatas: {inventario['duplicatas_exatas']} | pares quase-duplicados: {q['pares']} | limiar: {q['limiar']}")
print(f"Grupos com mais de um arquivo: {q['grupos_com_mais_de_um_arquivo']} ({q['imagens_em_grupos']} imagens); maior grupo: {q['maior_grupo']}")
print("Grupos que misturam classes:", q["grupos_entre_classes"] or "nenhum")
pares = pd.read_csv(ROOT / "dados/splits/quase_duplicatas.csv")
pares.sort_values("distancia_hamming").head(20)

In [ ]:
from projeto3.prepare import dhash_variants

hashes = []
for arquivo in particoes["arquivo"]:
    with Image.open(ROOT / arquivo) as imagem:
        hashes.append(dhash_variants(imagem))
hashes = np.stack(hashes)
original = hashes[:, 0, :]
vizinho = np.empty(len(hashes), dtype=int)
for inicio in range(0, len(hashes), 100):
    d = np.bitwise_count(original[inicio:inicio + 100, None, None, :] ^ hashes[None]).sum(-1).min(-1)
    d[np.arange(d.shape[0]), np.arange(inicio, inicio + d.shape[0])] = 10**6
    vizinho[inicio:inicio + 100] = d.min(axis=1)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(vizinho, bins=np.arange(0, vizinho.max() + 2) - 0.5, color="#4C72B0")
ax.axvline(q["limiar"] + 0.5, color="crimson", ls="--", label=f"limiar = {q['limiar']}")
ax.set_xlabel("distância de Hamming ao vizinho mais próximo (dHash 256 bits, 8 orientações)")
ax.set_ylabel("imagens")
ax.set_yscale("log")
ax.legend()
ax.set_title("Separação entre versões da mesma cena e imagens distintas")
salvar(fig, "eda_distancias_hash.png")
print(pd.Series(vizinho).describe())

In [ ]:
mostrar = pares.sort_values(["distancia_hamming", "arquivo_a"]).head(8)
if mostrar.empty:
    print("Nenhum par encontrado.")
else:
    fig, axes = plt.subplots(2, len(mostrar), figsize=(2.2 * len(mostrar), 5.4), squeeze=False)
    for coluna, (_, par) in enumerate(mostrar.iterrows()):
        for linha, lado in enumerate("ab"):
            with Image.open(ROOT / par[f"arquivo_{lado}"]) as imagem:
                axes[linha, coluna].imshow(imagem.convert("RGB"))
            nome = Path(par[f"arquivo_{lado}"]).name
            if lado == "b":
                nome += f"\nd = {par['distancia_hamming']}" + (" (idêntica)" if par["duplicata_exata"] else "")
            axes[linha, coluna].set_title(nome, fontsize=7)
            axes[linha, coluna].axis("off")
    fig.suptitle("Pares de duplicatas/quase-duplicatas (mantidos na mesma partição)")
    salvar(fig, "eda_duplicatas.png")

## 3. Verificação de sobreposição entre partições

Nenhum arquivo, conteúdo idêntico (SHA-256) ou grupo de quase-duplicatas pode aparecer em mais de uma partição.

In [ ]:
assert particoes["arquivo"].is_unique
por_hash = particoes.groupby("sha256")["particao"].nunique()
por_grupo = particoes.groupby("grupo")["particao"].nunique()
print("Hashes presentes em mais de uma partição:", int((por_hash > 1).sum()))
print("Grupos presentes em mais de uma partição:", int((por_grupo > 1).sum()))
assert (por_hash == 1).all() and (por_grupo == 1).all()

## 4. Exemplos das dez classes (treino)

In [ ]:
rng = np.random.default_rng(42)
n = 6
fig, axes = plt.subplots(len(CLASSES), n, figsize=(n * 1.7, len(CLASSES) * 1.75))
for i, classe in enumerate(CLASSES):
    arquivos = treino.loc[treino["classe"] == classe, "arquivo"].to_numpy()
    for j, arquivo in enumerate(rng.choice(arquivos, n, replace=False)):
        with Image.open(ROOT / arquivo) as imagem:
            axes[i, j].imshow(imagem.convert("RGB"))
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(NOMES_PT[classe], fontsize=8)
fig.suptitle("Exemplos aleatórios do treino por classe")
salvar(fig, "eda_exemplos.png")

## 5. Cor e textura por classe (treino)

Estatísticas simples por imagem: média de cada canal, desvio-padrão do cinza (contraste) e magnitude média do gradiente
(proxy de textura/bordas), calculadas na imagem reduzida a 128×128.

In [ ]:
def descritores(caminho):
    with Image.open(ROOT / caminho) as imagem:
        rgb = np.asarray(imagem.convert("RGB").resize((128, 128), Image.Resampling.BILINEAR), dtype=np.float32) / 255
    cinza = rgb.mean(axis=2)
    gy, gx = np.gradient(cinza)
    histograma = np.concatenate([np.histogram(rgb[..., c], bins=8, range=(0, 1))[0] for c in range(3)]) / (128 * 128)
    return rgb.mean(axis=(0, 1)), cinza.std(), np.hypot(gx, gy).mean(), histograma

valores = [descritores(a) for a in treino["arquivo"]]
estat = treino[["arquivo", "classe"]].copy()
estat[["R", "G", "B"]] = np.stack([v[0] for v in valores])
estat["contraste"] = [v[1] for v in valores]
estat["gradiente"] = [v[2] for v in valores]
histogramas = np.stack([v[3] for v in valores])
estat.groupby("classe")[["R", "G", "B", "contraste", "gradiente"]].mean().reindex(CLASSES).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
nomes = [NOMES_PT[c] for c in CLASSES]
for ax, (coluna, titulo) in zip(axes, [("G", "média do canal verde"), ("contraste", "desvio-padrão do cinza"),
                                       ("gradiente", "magnitude média do gradiente")]):
    ax.boxplot([estat.loc[estat["classe"] == c, coluna] for c in CLASSES], showfliers=False)
    ax.set_xticks(range(1, 11), nomes, rotation=60, ha="right", fontsize=8)
    ax.set_title(titulo)
    ax.grid(alpha=0.3)
salvar(fig, "eda_estatisticas.png")

## 6. Ambiguidades visuais (treino)

Para localizar ambiguidades de forma objetiva, cada imagem de treino é descrita por histogramas de cor (8 faixas por canal),
contraste e gradiente, e atribuída ao centróide de classe mais próximo (distância euclidiana nos descritores padronizados).
A matriz indica quais classes se misturam já em atributos globais simples; as imagens abaixo são as de treino mais próximas
do centróide de **outra** classe.

In [ ]:
X = np.column_stack([histogramas, estat[["contraste", "gradiente"]].to_numpy()])
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)
y = treino["classe"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()
centroides = np.stack([X[y == i].mean(axis=0) for i in range(len(CLASSES))])
dist = np.linalg.norm(X[:, None, :] - centroides[None], axis=2)
previsto = dist.argmin(axis=1)
matriz = np.zeros((10, 10), dtype=int)
np.add.at(matriz, (y, previsto), 1)
normalizada = matriz / matriz.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.imshow(normalizada, cmap="Blues", vmin=0, vmax=1)
for i in range(10):
    for j in range(10):
        if normalizada[i, j] >= 0.005:
            ax.text(j, i, f"{normalizada[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if normalizada[i, j] > 0.6 else "black")
ax.set_xticks(range(10), nomes, rotation=60, ha="right", fontsize=7)
ax.set_yticks(range(10), nomes, fontsize=7)
ax.set_xlabel("centróide mais próximo")
ax.set_ylabel("classe real")
ax.set_title("Centróide mais próximo em cor/textura (treino)")
salvar(fig, "eda_confusao_centroides.png")
pares_confusos = sorted(((normalizada[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(10) if i != j), reverse=True)[:8]
pd.DataFrame(pares_confusos, columns=["fração", "real", "centróide"]).round(3)

In [ ]:
margem = dist[np.arange(len(y)), y] - np.where(np.eye(10, dtype=bool)[y], np.inf, dist).min(axis=1)
ambiguas = np.argsort(-margem)[:10]
fig, axes = plt.subplots(2, 5, figsize=(12, 6.6))
for ax, indice in zip(axes.flat, ambiguas):
    with Image.open(ROOT / treino.loc[indice, "arquivo"]) as imagem:
        ax.imshow(imagem.convert("RGB"))
    outra = np.where(np.arange(10) == y[indice], np.inf, dist[indice]).argmin()
    ax.set_title(f"rótulo: {NOMES_PT[CLASSES[y[indice]]]}\nparece: {NOMES_PT[CLASSES[outra]]}", fontsize=8)
    ax.axis("off")
fig.suptitle("Imagens de treino mais próximas de outra classe em cor/textura")
salvar(fig, "eda_ambiguidades.png")

## 7. Efeito da resolução: a mesma imagem em 256×256 e 128×128

As redes recebem a imagem redimensionada **a partir do original** (bilinear com antialias). Na redução para 128×128 cada
pixel passa a cobrir 2×2 pixels originais; o recorte ampliado evidencia a perda em estruturas pequenas
(tanques, casas, aeronaves).

In [ ]:
escolhidas = ["storage_tank", "dense_residential", "airport", "harbor"]
fig, axes = plt.subplots(len(escolhidas), 4, figsize=(12, 3.1 * len(escolhidas)))
for linha, classe in enumerate(escolhidas):
    arquivo = treino.loc[treino["classe"] == classe, "arquivo"].iloc[0]
    with Image.open(ROOT / arquivo) as imagem:
        original = imagem.convert("RGB")
    reduzida = original.resize((128, 128), Image.Resampling.BILINEAR)
    recorte256 = original.crop((64, 64, 128, 128))
    recorte128 = reduzida.crop((32, 32, 64, 64))
    for ax, img, titulo in zip(axes[linha], [original, reduzida, recorte256, recorte128],
                               ["256×256 (original)", "128×128", "recorte 64×64 do original", "mesmo recorte em 128 (32×32)"]):
        ax.imshow(img, interpolation="nearest")
        ax.set_title(f"{NOMES_PT[classe]} — {titulo}", fontsize=8)
        ax.axis("off")
fig.suptitle("Mesma imagem em 256×256 e 128×128")
salvar(fig, "eda_resolucao.png")

## 8. Normalização

Entradas convertidas para RGB (3 canais), redimensionadas a partir do original e divididas por 255 → intervalo [0, 1].
Essa normalização não aprende estatística, portanto não há risco de vazamento. As médias e desvios abaixo, **calculados só
no treino**, são apenas descritivos. O aumento de dados (espelhamentos e giros de 90°) atua apenas no treino.

In [ ]:
medias = estat[["R", "G", "B"]].mean()
print("Média por canal no treino (pixels/255):", medias.round(4).to_dict())
amostra = np.stack([np.asarray(Image.open(ROOT / a).convert("RGB").resize((128, 128)), dtype=np.float32) / 255
                    for a in treino["arquivo"].iloc[::10]])
print("Desvio por canal (amostra de 1/10 do treino):", dict(zip("RGB", amostra.std(axis=(0, 1, 2)).round(4))))
print("Faixa após normalização:", float(amostra.min()), "a", float(amostra.max()))